# Gender and Parental Education Effects on Student Test Performance
## Statistical Analysis — Application Report for M.Sc. Data Science (Winter 2026/2027)

This notebook contains the complete analysis code for the application report. All descriptive statistics, assumption checks, hypothesis tests, post-hoc comparisons, and figures are reproduced below.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f'pandas {pd.__version__}')
print(f'numpy {np.__version__}')
print(f'scipy {stats.scipy.__version__}')

In [ ]:
# Load data — semicolon-separated, comma as decimal separator
df_long = pd.read_csv('Scores.csv', sep=';', decimal=',', index_col=0)
print(f'Long format: {df_long.shape[0]} rows, {df_long.shape[1]} columns')
df_long.head()

In [ ]:
# Reshape to wide format: one row per student
df = df_long.pivot_table(
    index=['student_id', 'gender', 'parental.level.of.education'],
    columns='subject',
    values='score'
).reset_index()
df.columns.name = None
df.rename(columns={'math': 'math_score', 'language': 'language_score'}, inplace=True)

print(f'Wide format: {df.shape[0]} students')
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

In [ ]:
# Verify counts
print('Gender counts:')
print(df['gender'].value_counts())
print('\nParental education counts:')
print(df['parental.level.of.education'].value_counts())

## 2. Descriptive Analysis

### 2.1 Descriptive Statistics by Gender (Table 1)

In [ ]:
def descriptive_stats(data, score_col):
    return pd.Series({
        'n': len(data),
        'Mean': round(data[score_col].mean(), 2),
        'Median': round(data[score_col].median(), 2),
        'SD': round(data[score_col].std(), 2),
        'IQR': round(data[score_col].quantile(0.75) - data[score_col].quantile(0.25), 2)
    })

print('=== Table 1: Descriptive Statistics by Gender ===')
for subject, col in [('Mathematics', 'math_score'), ('Language', 'language_score')]:
    print(f'\n{subject}:')
    for gender in ['female', 'male']:
        subset = df[df['gender'] == gender]
        s = descriptive_stats(subset, col)
        print(f'  {gender.capitalize():8s}  n={int(s["n"]):3d}  Mean={s["Mean"]:6.2f}  '
              f'Median={s["Median"]:6.2f}  SD={s["SD"]:5.2f}  IQR={s["IQR"]:5.2f}')

### 2.2 Descriptive Statistics by Parental Education (Table 2)

In [ ]:
edu_order = ['high school', "associate's degree", "bachelor's degree", "master's degree"]

print('=== Table 2: Descriptive Statistics by Parental Education ===')
for subject, col in [('Mathematics', 'math_score'), ('Language', 'language_score')]:
    print(f'\n{subject}:')
    for edu in edu_order:
        subset = df[df['parental.level.of.education'] == edu]
        s = descriptive_stats(subset, col)
        print(f'  {edu:22s}  n={int(s["n"]):3d}  Mean={s["Mean"]:6.2f}  '
              f'Median={s["Median"]:6.2f}  SD={s["SD"]:5.2f}  IQR={s["IQR"]:5.2f}')

### 2.3 Figure 1: Boxplot by Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

for ax, (subject, col) in zip(axes, [('Mathematics', 'math_score'), ('Language', 'language_score')]):
    sns.boxplot(data=df, x='gender', y=col, order=['female', 'male'],
                palette=['#4C72B0', '#DD8452'], ax=ax, showmeans=True,
                meanprops={'marker': 'D', 'markerfacecolor': 'black',
                           'markeredgecolor': 'black', 'markersize': 6})
    ax.set_title(f'{subject} Scores by Gender', fontsize=12)
    ax.set_xlabel('Gender', fontsize=11)
    ax.set_ylabel('Score' if ax == axes[0] else '', fontsize=11)
    ax.set_xticklabels(['Female', 'Male'])

plt.tight_layout()
plt.savefig('figure1_gender_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figure1_gender_boxplot.png')

### 2.4 Figure 2: Boxplot by Parental Education

In [ ]:
edu_labels = ['High School', "Associate's\nDegree", "Bachelor's\nDegree", "Master's\nDegree"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, (subject, col) in zip(axes, [('Mathematics', 'math_score'), ('Language', 'language_score')]):
    sns.boxplot(data=df, x='parental.level.of.education', y=col,
                order=edu_order, palette='Blues', ax=ax, showmeans=True,
                meanprops={'marker': 'D', 'markerfacecolor': 'black',
                           'markeredgecolor': 'black', 'markersize': 6})
    ax.set_title(f'{subject} Scores by Parental Education', fontsize=12)
    ax.set_xlabel('Parental Level of Education', fontsize=11)
    ax.set_ylabel('Score' if ax == axes[0] else '', fontsize=11)
    ax.set_xticklabels(edu_labels, fontsize=9)

plt.tight_layout()
plt.savefig('figure2_education_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figure2_education_boxplot.png')

## 3. Inferential Analysis

### 3.1 Assumption Checks

In [ ]:
# --- Shapiro-Wilk tests for gender subgroups ---
print('=== Shapiro-Wilk Tests (Gender Subgroups) ===')
for gender in ['female', 'male']:
    subset = df[df['gender'] == gender]
    for col, label in [('math_score', 'Mathematics'), ('language_score', 'Language')]:
        w, p = stats.shapiro(subset[col])
        print(f'  {gender.capitalize()} {label}: W = {w:.4f}, p = {p:.4f}')

print()

# --- Levene's tests for gender ---
print('=== Levene\'s Tests (Gender) ===')
female = df[df['gender'] == 'female']
male = df[df['gender'] == 'male']

for col, label in [('math_score', 'Mathematics'), ('language_score', 'Language')]:
    f_stat, p_val = stats.levene(female[col], male[col], center='median')
    print(f'  {label}: F = {f_stat:.4f}, p = {p_val:.4f}')

In [ ]:
# --- Shapiro-Wilk tests for parental education subgroups ---
print('=== Shapiro-Wilk Tests (Parental Education Subgroups) ===')
for edu in edu_order:
    subset = df[df['parental.level.of.education'] == edu]
    for col, label in [('math_score', 'Mathematics'), ('language_score', 'Language')]:
        w, p = stats.shapiro(subset[col])
        flag = ' ***' if p < 0.05 else ''
        print(f'  {edu:22s} {label}: W = {w:.4f}, p = {p:.4f}{flag}')

print()

# --- Levene's test for parental education (Language only, since Math uses Kruskal-Wallis) ---
print('=== Levene\'s Test (Parental Education — Language) ===')
groups_lang = [df[df['parental.level.of.education'] == edu]['language_score'] for edu in edu_order]
f_stat, p_val = stats.levene(*groups_lang, center='median')
print(f'  F = {f_stat:.4f}, p = {p_val:.4f}')

### 3.2 RQ1: Gender Differences (Two-Sample t-Test)

In [ ]:
print('=== RQ1: Two-Sample t-Tests (Gender) ===')

for col, label in [('math_score', 'Mathematics'), ('language_score', 'Language')]:
    t_stat, p_val = stats.ttest_ind(female[col], male[col], equal_var=True)
    
    # Cohen's d
    n1, n2 = len(female), len(male)
    s1, s2 = female[col].std(), male[col].std()
    pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    d = (female[col].mean() - male[col].mean()) / pooled_std
    
    print(f'\n{label}:')
    print(f'  t({n1 + n2 - 2}) = {t_stat:.4f}')
    print(f'  p-value = {p_val:.6f}')
    print(f'  Cohen\'s d = {d:.3f}')
    print(f'  Female mean = {female[col].mean():.2f}, Male mean = {male[col].mean():.2f}')

### 3.3 RQ2: Parental Education — Mathematics (Kruskal-Wallis + Dunn's Test)

In [ ]:
print('=== RQ2 Mathematics: Kruskal-Wallis Test ===')
groups_math = [df[df['parental.level.of.education'] == edu]['math_score'] for edu in edu_order]
h_stat, p_val = stats.kruskal(*groups_math)
print(f'  H = {h_stat:.4f}, df = 3, p = {p_val:.6f}')

In [ ]:
# Dunn's post-hoc test with Bonferroni correction
print('=== Post-Hoc: Dunn\'s Test with Bonferroni Correction (Mathematics) ===')
dunn_result = sp.posthoc_dunn(
    df, val_col='math_score', group_col='parental.level.of.education',
    p_adjust='bonferroni'
)

print('\nBonferroni-adjusted p-values (compared against alpha = 0.05):')
print(dunn_result.round(4))

# Print pairwise results for the report table
print('\n--- Table 3 values ---')
pairs = [
    ('high school', "associate's degree"),
    ('high school', "bachelor's degree"),
    ('high school', "master's degree"),
    ("associate's degree", "bachelor's degree"),
    ("associate's degree", "master's degree"),
    ("bachelor's degree", "master's degree")
]
for g1, g2 in pairs:
    p_adj = dunn_result.loc[g1, g2]
    sig = 'Yes' if p_adj < 0.05 else 'No'
    print(f'  {g1:22s} vs {g2:22s}: p_adj = {p_adj:.4f}  Significant: {sig}')

In [ ]:
# Also compute raw p-values for completeness (Table 3 includes both)
print('=== Raw (unadjusted) Dunn p-values ===')
dunn_raw = sp.posthoc_dunn(
    df, val_col='math_score', group_col='parental.level.of.education',
    p_adjust='none'
)
for g1, g2 in pairs:
    p_raw = dunn_raw.loc[g1, g2]
    p_adj = dunn_result.loc[g1, g2]
    print(f'  {g1:22s} vs {g2:22s}: p_raw = {p_raw:.4f}, p_adj = {p_adj:.4f}')

### 3.4 RQ2: Parental Education — Language (One-Way ANOVA + Tukey's HSD)

In [ ]:
print('=== RQ2 Language: One-Way ANOVA ===')
f_stat, p_val = stats.f_oneway(*groups_lang)
print(f'  F(3, 482) = {f_stat:.4f}, p = {p_val:.6e}')

In [ ]:
# Tukey's HSD post-hoc test
print('=== Post-Hoc: Tukey\'s HSD (Language) ===')
tukey = pairwise_tukeyhsd(
    df['language_score'],
    df['parental.level.of.education'],
    alpha=0.05
)
print(tukey)

# Extract MS_within for the report
n_total = len(df)
k = 4
grand_mean = df['language_score'].mean()
ss_within = sum(
    ((df[df['parental.level.of.education'] == edu]['language_score'] - 
      df[df['parental.level.of.education'] == edu]['language_score'].mean())**2).sum()
    for edu in edu_order
)
ms_within = ss_within / (n_total - k)
print(f'\nMS_within = {ms_within:.2f}, df_within = {n_total - k}')

## 4. Summary of All Results

All values above should match the report tables and text. Key results:

- **RQ1 Math**: t(484) = -3.78, p < 0.001, d = -0.34 (males score higher)
- **RQ1 Lang**: t(484) = 6.41, p < 0.001, d = 0.58 (females score higher)
- **RQ2 Math**: H = 15.68, p = 0.0013 (high school group significantly lower)
- **RQ2 Lang**: F(3,482) = 14.24, p < 0.0001 (high school group significantly lower)